## Objectives

- Analyze Canada's economic indicators during the 2025–2026 technical recession.
- Identify factors influencing GDP growth.
- Explore relationships between inflation, unemployment, interest rates, and GDP.
- Build machine learning models to predict economic growth and recession risk.
- Visualize findings through an interactive dashboard.
- Generate actionable insights from economic data.


## Installing Required Libraries

In [1]:
!pip install numpy==2.2.0
!pip install pandas==2.2.3
!pip install scikit-learn==1.6.0
!pip install matplotlib==3.9.3
!pip install seaborn==0.13.2


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import statsmodels.api as sm
import plotly.express as px


print("All libraries installed successfully!")

All libraries installed successfully!


## Loading Data

In [3]:
df_gdp = pd.read_csv(r"C:\Users\twins\OneDrive\Desktop\Data\project\CANADA\36100434-eng\36100434.csv" ,low_memory=False)
df_ir = pd.read_csv(r"C:\Users\twins\OneDrive\Desktop\Data\project\CANADA\bank_rate.csv",skiprows=11,sep=",",low_memory=False)
df_unemp = pd. read_csv(r"C:\Users\twins\OneDrive\Desktop\Data\project\CANADA\14100287-eng_UNEMPLOYMENT\14100287.csv", low_memory=False)
df_cpi = pd.read_csv(r"C:\Users\twins\OneDrive\Desktop\Data\project\CANADA\18100004-eng_CPI\18100004.csv", skiprows=0,low_memory=False)
df_retail = pd.read_csv(r"C:\Users\twins\OneDrive\Desktop\Data\project\CANADA\retail_databaseLoadingData.csv", low_memory=False)

In [4]:
#verify the files loaded properly.

print("GDP:", df_gdp.shape)
print("Interest Rate:", df_ir.shape)
print("Unemployment:", df_unemp.shape)
print("CPI:", df_cpi.shape)
print("Retail:", df_retail.shape)


GDP: (262197, 17)
Interest Rate: (2608, 2)
Unemployment: (5430564, 19)
CPI: (1144893, 15)
Retail: (574, 17)


## Inspecting Each Datasets

In [5]:
print("\nGDP Columns")
print(df_gdp.columns)

print("\nInterest Rate Columns")
print(df_ir.columns)

print("\nUnemployment Columns")
print(df_unemp.columns)

print("\nCPI Columns")
print(df_cpi.columns)

print("\nRetail Columns")
print(df_retail.columns)


GDP Columns
Index(['REF_DATE', 'GEO', 'DGUID', 'Seasonal adjustment', 'Prices',
       'North American Industry Classification System (NAICS)', 'UOM',
       'UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE',
       'STATUS', 'SYMBOL', 'TERMINATED', 'DECIMALS'],
      dtype='object')

Interest Rate Columns
Index(['Date', 'V39079'], dtype='object')

Unemployment Columns
Index(['REF_DATE', 'GEO', 'DGUID', 'Labour force characteristics', 'Gender',
       'Age group', 'Statistics', 'Data type', 'UOM', 'UOM_ID',
       'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE', 'STATUS',
       'SYMBOL', 'TERMINATED', 'DECIMALS'],
      dtype='object')

CPI Columns
Index(['REF_DATE', 'GEO', 'DGUID', 'Products and product groups', 'UOM',
       'UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE',
       'STATUS', 'SYMBOL', 'TERMINATED', 'DECIMALS'],
      dtype='object')

Retail Columns
Index(['REF_DATE', 'GEO', 'DGUID',
       'North American In

## Standardize column name

In [6]:
def clean_dataset(df):
    df = df.copy()

# remove duplicate columns (IMPORTANT)
    df = df.loc[:, ~df.columns.duplicated()]

 # standardize names
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

 # rename core columns
    df.rename(columns={"ref_date": "date","geo": "geo","value": "value"
    }, inplace=True)

# fix interest rate case
    if "v39079" in df.columns:
        df.rename(columns={"v39079": "value"}, inplace=True)

 # convert date
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce") 
       
# ensure numeric value
    if "value" in df.columns:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")

    return df

In [7]:

df_gdp_clean = clean_dataset(df_gdp)
df_unemp_clean = clean_dataset(df_unemp)
df_cpi_clean = clean_dataset(df_cpi)
df_retail_clean = clean_dataset(df_retail)
df_ir_clean = clean_dataset(df_ir)

In [8]:
for name, df in {
    "GDP": df_gdp_clean,
    "UNEMP": df_unemp_clean,
    "CPI": df_cpi_clean,
    "RETAIL": df_retail_clean,
    "Interest Rate":df_ir_clean
}.items():
    print("\n", name)
    print(df.head())
    print(df.dtypes)


 GDP
        date     geo           dguid                  seasonal_adjustment  \
0 1997-01-01  Canada  2021A000011124  Seasonally adjusted at annual rates   
1 1997-01-01  Canada  2021A000011124  Seasonally adjusted at annual rates   
2 1997-01-01  Canada  2021A000011124  Seasonally adjusted at annual rates   
3 1997-01-01  Canada  2021A000011124  Seasonally adjusted at annual rates   
4 1997-01-01  Canada  2021A000011124  Seasonally adjusted at annual rates   

                   prices  \
0  Chained (2017) dollars   
1  Chained (2017) dollars   
2  Chained (2017) dollars   
3  Chained (2017) dollars   
4  Chained (2017) dollars   

  north_american_industry_classification_system_(naics)      uom  uom_id  \
0                              All industries [T001]     Dollars      81   
1                  Goods-producing industries [T002]     Dollars      81   
2               Services-producing industries [T003]     Dollars      81   
3                  Business sector industries [T004]

In [9]:
#monthly alignment

def to_monthly(df):
    df = df.copy()

#ensure datetime index
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")

#convert to datetime 
    df["year"] = df_retail_clean["date"].dt.year


#keep ONLY numeric columns before resampling
    df = df.select_dtypes(include="number")

    #monthly aggregation
    df_monthly = df.resample("ME").mean()
    df_monthly = df_monthly.reset_index()

    return df_monthly

In [10]:

datasets = {
    "gdp": df_gdp_clean,
    "unemp": df_unemp_clean,
    "cpi": df_cpi_clean,
    "retail": df_retail_clean,
    "ir": df_ir_clean
}

cleaned = {}

for name, df in datasets.items():
    df_clean = clean_dataset(df)
    df_monthly = to_monthly(df_clean)

#rename value column to avoid collisions later
    df_monthly = df_monthly.rename(columns={"value": name})

    cleaned[name] = df_monthly

In [11]:
# inspect result

for name, df in cleaned.items():
   print(name)
   print(df.head())
   print(df.dtypes)
   print(df.isna().sum())
   print("\n")

gdp
        date  uom_id  scalar_id           gdp  symbol  terminated  decimals  \
0 1997-01-31    81.0        6.0  25309.792793     NaN         NaN       0.0   
1 1997-02-28    81.0        6.0  25513.918919     NaN         NaN       0.0   
2 1997-03-31    81.0        6.0  25486.756757     NaN         NaN       0.0   
3 1997-04-30    81.0        6.0  25669.479730     NaN         NaN       0.0   
4 1997-05-31    81.0        6.0  25743.099099     NaN         NaN       0.0   

   year  
0   NaN  
1   NaN  
2   NaN  
3   NaN  
4   NaN  
date          datetime64[ns]
uom_id               float64
scalar_id            float64
gdp                  float64
symbol               float64
terminated           float64
decimals             float64
year                 float64
dtype: object
date            0
uom_id          0
scalar_id       0
gdp             0
symbol        351
terminated    351
decimals        0
year          351
dtype: int64


unemp
        date      uom_id  scalar_id       unemp  s

In [12]:
df_ir_clean["date"] = pd.to_datetime(df_ir_clean["date"])

df_ir_clean = df_ir_clean.set_index("date").resample("ME").mean().reset_index()

for df in [df_gdp_clean, df_unemp_clean, df_cpi_clean, df_retail_clean, df_ir_clean]:
    df["date"] = pd.to_datetime(df["date"])
    df["date"] = df["date"].dt.to_period("M").dt.to_timestamp("M")


In [13]:
df_ir_clean

,date,value
0,2016-05-31,0.50
1,2016-06-30,0.50
2,2016-07-31,0.50
3,2016-08-31,0.50
4,2016-09-30,0.50
...,...,...
116,2026-01-31,2.25
117,2026-02-28,2.25
118,2026-03-31,2.25
119,2026-04-30,2.25


In [14]:
for df in [df_gdp_clean, df_ir_clean, df_unemp_clean, df_cpi_clean, df_retail_clean]:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

In [15]:
start_date = "2024-01-01"

df_gdp_f = df_gdp_clean[df_gdp_clean["date"] >= start_date]
df_ir_f= df_ir_clean[df_ir_clean["date"] >= start_date]
df_unemp_f = df_unemp_clean[df_unemp_clean["date"] >= start_date]
df_cpi_f = df_cpi_clean[df_cpi_clean["date"] >= start_date]
df_retail_f = df_retail_clean[df_retail_clean["date"] >= start_date]

In [16]:
for name, df in {
    "gdp": df_gdp_f,
    "ir": df_ir_f,
    "unemp": df_unemp_f,
    "cpi": df_cpi_f,
    "retail": df_retail_f
}.items():
    print(name)
    print(df["date"].min(), df["date"].max())
    print(df.shape)
    print("-"*40)

gdp
2024-01-31 00:00:00 2026-03-31 00:00:00
(20169, 17)
----------------------------------------
ir
2024-01-31 00:00:00 2026-05-31 00:00:00
(29, 2)
----------------------------------------
unemp
2024-01-31 00:00:00 2026-04-30 00:00:00
(251748, 19)
----------------------------------------
cpi
2024-01-31 00:00:00 2026-04-30 00:00:00
(56095, 15)
----------------------------------------
retail
2024-01-31 00:00:00 2026-03-31 00:00:00
(378, 17)
----------------------------------------


In [17]:
# Renaming value to the desired economic variable

df_gdp_m = df_gdp_f.rename(columns={"value": "gdp"})
df_unemp_m = df_unemp_f.rename(columns={"value": "unemployment"})
df_cpi_m = df_cpi_f.rename(columns={"value": "cpi"})
df_retail_m = df_retail_f.rename(columns={"value": "retail"})
df_ir_m = df_ir_f.rename(columns={"value": "interest_rate"})

In [18]:
# grouping data in terms of month

df_gdp_cleaned = df_gdp_m.groupby("date", as_index=False)["gdp"].mean()
df_unemp_cleaned = df_unemp_m.groupby("date", as_index=False)["unemployment"].mean()
df_cpi_cleaned = df_cpi_m.groupby("date", as_index=False)["cpi"].mean()
df_retail_cleaned = df_retail_m.groupby("date", as_index=False)["retail"].mean()
df_ir_cleaned = df_ir_m.groupby("date", as_index=False)["interest_rate"].mean()

In [19]:
df_gdp_m

,date,geo,dguid,seasonal_adjustment,prices,north_american_industry_classification_system_(naics),uom,uom_id,scalar_factor,scalar_id,vector,coordinate,gdp,status,symbol,terminated,decimals
242028,2024-01-31,Canada,2021A000011124,Seasonally adjusted at annual rates,Chained (2017) dollars,All industries [T001],Dollars,81,millions,6,v65201210,1.1.1.1,2262764.0,NaN,NaN,NaN,0
242029,2024-01-31,Canada,2021A000011124,Seasonally adjusted at annual rates,Chained (2017) dollars,Goods-producing industries [T002],Dollars,81,millions,6,v65201211,1.1.1.2,572038.0,NaN,NaN,NaN,0
242030,2024-01-31,Canada,2021A000011124,Seasonally adjusted at annual rates,Chained (2017) dollars,Services-producing industries [T003],Dollars,81,millions,6,v65201212,1.1.1.3,1697917.0,NaN,NaN,NaN,0
242031,2024-01-31,Canada,2021A000011124,Seasonally adjusted at annual rates,Chained (2017) dollars,Business sector industries [T004],Dollars,81,millions,6,v65201213,1.1.1.4,1837378.0,NaN,NaN,NaN,0
242032,2024-01-31,Canada,2021A000011124,Seasonally adjusted at annual rates,Chained (2017) dollars,Non-business sector industries [T007],Dollars,81,millions,6,v65201216,1.1.1.7,425715.0,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
262192,2026-03-31,Canada,2021A000011124,Trading-day adjusted,2017 constant prices,Defence services [9111],Dollars,81,millions,6,v65202024,1.2.2.269,1477.0,NaN,NaN,NaN,0
262193,2026-03-31,Canada,2021A000011124,Trading-day adjusted,2017 constant prices,Federal government public administration (exce...,Dollars,81,millions,6,v65202025,1.2.2.270,3826.0,NaN,NaN,NaN,0
262194,2026-03-31,Canada,2021A000011124,Trading-day adjusted,2017 constant prices,Provincial and territorial public administrati...,Dollars,81,millions,6,v65202026,1.2.2.271,3596.0,NaN,NaN,NaN,0
262195,2026-03-31,Canada,2021A000011124,Trading-day adjusted,2017 constant prices,"Local, municipal and regional public administr...",Dollars,81,millions,6,v65202027,1.2.2.272,4604.0,NaN,NaN,NaN,0


In [20]:
# keeping only the columns actually need

df_gdp_final = df_gdp_cleaned[["date","gdp"]]
df_unemp_final = df_unemp_cleaned[["date","unemployment"]]
df_cpi_final = df_cpi_cleaned[["date","cpi"]]
df_retail_final = df_retail_cleaned[["date","retail"]]
df_ir_final = df_ir_cleaned[["date","interest_rate"]]

In [21]:
#merging dataset

df = df_gdp_final.merge(df_unemp_final, on="date", how="outer") \
          .merge(df_cpi_final, on="date", how="outer") \
          .merge(df_retail_final, on="date", how="outer") \
          .merge(df_ir_final, on="date", how="outer")
df = df.sort_values("date").reset_index(drop=True)

In [22]:
# inspecting the result
print(df.shape)
print(df.head())

(29, 6)
        date           gdp  unemployment         cpi        retail  \
0 2024-01-31  39549.273092    351.252441  160.826734  9.584294e+06   
1 2024-02-29  39812.713521    352.719008  161.168844  9.594484e+06   
2 2024-03-31  39809.680054    353.983113  161.640251  9.522134e+06   
3 2024-04-30  39970.524766    355.413925  161.712170  9.606926e+06   
4 2024-05-31  40045.037483    357.988055  162.602843  9.554558e+06   

   interest_rate  
0            5.0  
1            5.0  
2            5.0  
3            5.0  
4            5.0  


In [23]:
print(df_gdp_final.head())
print(df_gdp_final.head())

        date           gdp
0 2024-01-31  39549.273092
1 2024-02-29  39812.713521
2 2024-03-31  39809.680054
3 2024-04-30  39970.524766
4 2024-05-31  40045.037483
        date           gdp
0 2024-01-31  39549.273092
1 2024-02-29  39812.713521
2 2024-03-31  39809.680054
3 2024-04-30  39970.524766
4 2024-05-31  40045.037483


In [24]:
df = df.sort_values("date").reset_index(drop=True)

In [25]:
df

,date,gdp,unemployment,cpi,retail,interest_rate
0,2024-01-31,39549.273092,351.252441,160.826734,9.584294e+06,5.000000
1,2024-02-29,39812.713521,352.719008,161.168844,9.594484e+06,5.000000
2,2024-03-31,39809.680054,353.983113,161.640251,9.522134e+06,5.000000
3,2024-04-30,39970.524766,355.413925,161.712170,9.606926e+06,5.000000
4,2024-05-31,40045.037483,357.988055,162.602843,9.554558e+06,5.000000
5,2024-06-30,40166.767068,359.093982,162.808628,9.491665e+06,4.787500
6,2024-07-31,40186.337349,359.344978,163.792120,9.683835e+06,4.695652
7,2024-08-31,40297.599732,360.541829,163.313217,9.657746e+06,4.500000
8,2024-09-30,40469.361446,359.210581,162.524489,9.760181e+06,4.285714
9,2024-10-31,40544.641232,359.676727,162.750773,9.863124e+06,4.119565


In [26]:
df = df.sort_values("date")

df = df.ffill()

In [27]:
df

,date,gdp,unemployment,cpi,retail,interest_rate
0,2024-01-31,39549.273092,351.252441,160.826734,9.584294e+06,5.000000
1,2024-02-29,39812.713521,352.719008,161.168844,9.594484e+06,5.000000
2,2024-03-31,39809.680054,353.983113,161.640251,9.522134e+06,5.000000
3,2024-04-30,39970.524766,355.413925,161.712170,9.606926e+06,5.000000
4,2024-05-31,40045.037483,357.988055,162.602843,9.554558e+06,5.000000
5,2024-06-30,40166.767068,359.093982,162.808628,9.491665e+06,4.787500
6,2024-07-31,40186.337349,359.344978,163.792120,9.683835e+06,4.695652
7,2024-08-31,40297.599732,360.541829,163.313217,9.657746e+06,4.500000
8,2024-09-30,40469.361446,359.210581,162.524489,9.760181e+06,4.285714
9,2024-10-31,40544.641232,359.676727,162.750773,9.863124e+06,4.119565


In [28]:
df.columns

Index(['date', 'gdp', 'unemployment', 'cpi', 'retail', 'interest_rate'], dtype='object')

In [29]:
# Flag rows where key indicators are duplicated (likely not yet released)
df["is_provisional"] = (
    df[["gdp", "retail", "unemployment"]].duplicated(keep=False)
)

# Separate confirmed vs provisional
df_confirmed   = df[~df["is_provisional"]].copy()
df_provisional = df[df["is_provisional"]].copy()

print(f"Confirmed rows:   {len(df_confirmed)}")
print(f"Provisional rows: {len(df_provisional)}")

Confirmed rows:   27
Provisional rows: 2


In [30]:
# Work only with confirmed data for analysis & modeling
df = df_confirmed.copy()
print(df.shape)  # Should be (27, 7)

(27, 7)
